# Recommendation System Training (Association Rules)

This notebook trains a **recommendation model** using synthetic transaction data.

### Objectives
- Simulate user purchase behavior using mock data
- Train an association rule model using Apriori
- Generate product recommendations
- Export the trained model for use in Django backend

### Output
- `recommendation.pkl` (trained model)

In [ ]:
import json
import pandas as pd
import pickle
from mlxtend.frequent_patterns import apriori, association_rules

##  Load Mock Transaction Data

We use synthetic transaction data where each row represents a customer order.

Example: ["Fresh Tomato", "Cavendish Banana"]

This replaces the need for a live database during model development.


In [ ]:
with open("data/mock_transactions.json", "r") as f:
    transactions = json.load(f)

transactions[:5]

## Data Preprocessing

We convert transactions into a **one-hot encoded format** required by the Apriori algorithm.

Each column = product  
Each row = transaction  
Value = True/False (whether product is present)

In [ ]:
def preprocess(transactions):
    all_items = sorted(set(item for t in transactions for item in t))

    encoded = []
    for t in transactions:
        row = {item: (item in t) for item in all_items}
        encoded.append(row)

    return pd.DataFrame(encoded)

df = preprocess(transactions)
df.head()

## Train Recommendation Model

We apply:
- **Apriori Algorithm** → find frequent itemsets
- **Association Rules** → generate relationships between products

### Key Metrics:
- **Support**: frequency of itemset
- **Confidence**: likelihood of purchase
- **Lift**: strength of association

In [ ]:
freq = apriori(df, min_support=0.01, use_colnames=True)
rules = association_rules(freq, metric="lift", min_threshold=1.0)

rules.head()

## Inspect Learned Rules

These rules represent patterns such as:

"If a user buys A → they are likely to buy B"

We sort by **lift** to find the strongest relationships.

In [ ]:
rules.sort_values(by="lift", ascending=False)[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].head(10)

## Recommendation Function

This function takes:
- A trained rule set
- A list of user-selected items

Returns:
- Recommended products not already selected

In [ ]:
def recommend(rules, user_items):
    recs = set()

    for _, row in rules.iterrows():
        if set(row["antecedents"]).issubset(user_items):
            recs.update(row["consequents"])

    return list(recs - set(user_items))

## Test Recommendations

We simulate a user selecting a product and observe suggested items.

In [ ]:
recommend(rules, {"Fresh Tomato"})

## Save Model

The trained model is exported as a `.pkl` file.

This file will be:
- Loaded in Django backend
- Used for real-time recommendations

In [ ]:
with open("recommendation.pkl", "wb") as f:
    pickle.dump(rules, f)

print("Model saved as recommendation.pkl")

## Load Model (Simulation)

This step simulates how the Django backend will load and use the model.

In [ ]:
with open("recommendation.pkl", "rb") as f:
    loaded_rules = pickle.load(f)

recommend(loaded_rules, {"Fresh Tomato"})

## Model Insights (Explainable AI)

To improve transparency, we analyse rule strength using Lift distribution.

In [ ]:
import matplotlib.pyplot as plt

rules["lift"].hist(bins=20)
plt.title("Lift Distribution")
plt.xlabel("Lift")
plt.ylabel("Frequency")
plt.show()